# PG vector - step by step

A hands-on lab. Run one cell at a time, read what happened, then move on. You will create the pgvector extension, add embeddings to instruments, populate them with deterministic vectors, run a similarity search to find comparable instruments, and create an HNSW index for faster lookups at scale.

**Before you start:** finish `00-setup/` so `LB_PROFILE`, `LB_PROJECT`, `LB_BRANCH`, `LB_SCHEMA` are set and the bank is loaded on your branch.

## Setup - initialize

In [ ]:
import os, sys, hashlib, struct
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'common', 'scripts'))
from lakebase import connect, run_sql

def hash_to_embedding(text: str, dim: int = 768) -> str:
    """Deterministic hash-based embedding for reproducibility."""
    h = hashlib.sha256(text.encode()).digest()
    values = []
    for i in range(dim):
        byte_idx = (i * 2) % len(h)
        val = (struct.unpack("B", h[byte_idx:byte_idx+1])[0] / 255.0) * 2 - 1
        values.append(f"{val:.4f}")
    return "[" + ",".join(values) + "]"

print('pgvector initialized')

## Step 1 - Check the instrument table

Query the instrument table to see what we're working with. The SQL:

```sql
SELECT count(*) FROM bank.instrument;
SELECT instrument_id, name FROM bank.instrument LIMIT 5;
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT count(*) FROM bank.instrument")
        count = cur.fetchone()[0]
        cur.execute("SELECT instrument_id, name FROM bank.instrument LIMIT 5")
        instruments = cur.fetchall()

print(f'Checking instrument table:')
print(f'  Total instruments: {count}')
print(f'  Sample instruments:')
for p in instruments:
    print(f'    - {p[0]}: {p[1]}')

## Step 2 - Create pgvector extension

Enable the pgvector extension for embeddings. The SQL:

```sql
CREATE EXTENSION IF NOT EXISTS vector;
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute("CREATE EXTENSION IF NOT EXISTS vector")
        print(f'pgvector extension created')

## Step 3 - Add embedding column to instrument table

Add a 768-dimensional vector column. The SQL:

```sql
ALTER TABLE bank.instrument ADD COLUMN IF NOT EXISTS embedding vector(768);
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute("ALTER TABLE bank.instrument ADD COLUMN IF NOT EXISTS embedding vector(768)")
        print(f'Embedding column added to bank.instrument')

## Step 4 - Populate embeddings

Generate embeddings from instrument names and specs (deterministic hashing for reproducibility). Then update the table. The SQL:

```sql
UPDATE bank.instrument SET embedding = CAST('<vector_value>' AS vector) WHERE instrument_id = '<id>';
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT instrument_id, name, spec FROM bank.instrument ORDER BY instrument_id")
        instruments = cur.fetchall()

print(f'Populating {len(instruments)} embeddings:')

with connect() as conn:
    with conn.cursor() as cur:
        for instrument_id, name, spec in instruments:
            # Generate embedding from name+spec
            text_for_embedding = f"{name} {spec}"
            embedding_str = hash_to_embedding(text_for_embedding, 768)
            cur.execute(
                f"UPDATE bank.instrument SET embedding = CAST(%s AS vector) WHERE instrument_id = %s",
                [embedding_str, instrument_id]
            )

print(f'  Updated {len(instruments)} instruments with embeddings')

## Step 5 - Run similarity search

Find similar instruments using cosine distance. The SQL:

```sql
SELECT instrument_id, name, embedding <-> embedding_query AS distance
FROM bank.instrument
WHERE instrument_id != 'INS-QTZL'
ORDER BY embedding <-> embedding_query
LIMIT 5;
```

In [ ]:
# The reference instrument and its stored embedding (same as pg_vector.py)
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT instrument_id, name, embedding::text FROM bank.instrument WHERE instrument_id = %s",
                    ["INS-QTZL"])
        query_part = cur.fetchone()
        query_part_id = query_part[0]
        query_part_name = query_part[1]

query_embedding = query_part[2]

# Find similar instruments
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(
            f"SELECT instrument_id, name, embedding <-> CAST(%s AS vector) AS distance "
            f"FROM bank.instrument "
            f"WHERE instrument_id != %s "
            f"ORDER BY embedding <-> CAST(%s AS vector) "
            f"LIMIT 5",
            [query_embedding, query_part_id, query_embedding]
        )
        similar_parts = cur.fetchall()

print(f'Query instrument: {query_part_id} ({query_part_name})')
print(f'\nMost similar comparable instruments:')
for instrument_id, name, distance in similar_parts:
    print(f'  {instrument_id}: {name} (distance: {distance:.4f})')

## Step 6 - Create HNSW index for scale

For fast similarity search at scale, create an HNSW index. The SQL:

```sql
CREATE INDEX IF NOT EXISTS idx_part_embedding_hnsw
ON bank.instrument USING hnsw (embedding vector_cosine_ops);
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(
            f"CREATE INDEX IF NOT EXISTS idx_part_embedding_hnsw "
            f"ON bank.instrument USING hnsw (embedding vector_cosine_ops)"
        )
        print(f'HNSW index created on bank.instrument.embedding')
        print(f'\nIndex optimization: HNSW (Hierarchical Navigable Small World)')
        print(f'  Fast approximate nearest neighbor search')
        print(f'  O(log n) query time for millions of vectors')

## What you learned

You added pgvector to Lakebase: created embeddings from operational data (instrument specs), ran similarity search to rank comparable instruments, and indexed with HNSW for production scale. All on live data, no separate vector database needed. (The workshop embeds with a reproducible hash, so the rankings are not financially meaningful; swap in a real embedding model for semantic neighbours.) The pattern extends across the bank: similar counterparties, similar funds by holdings profile, similar transactions for fraud clustering - operational vectors stored in the same Postgres table as your transaction data.

**Next:** the branching, time-travel, and performance journeys.